# Resolution Control: stating how your instrument smears

Every SANS measurement is convolved with the instrument's resolution. A fitted
radius therefore only means something alongside the smearing that produced it,
so SANS-fitter treats resolution as a **stated choice**, not as a property of
whichever file you happened to load.
The four modes mirror SasView's Fit Page — *None* / *Use dQ Data* /
*Custom Pinhole* / *Custom Slit*:

```python
fitter.set_resolution('data')                      # default: the file's own columns
fitter.set_resolution('none')                      # perfect resolution
fitter.set_resolution('pinhole', dq_over_q=0.10)   # constant relative width
fitter.set_resolution('slit', slit_length=0.05)    # constant slit geometry
```

This notebook demonstrates:

1. Reading the active setting, and what the default actually does
2. What smearing does to a measurement
3. What each mode hands sasmodels
4. What ignoring resolution costs you, measured against known truth
5. Stating a width for a file that carries no `dQ` column, and slit (USANS) geometry
6. Provenance: your dataset is never modified, and the choice is recorded

In [ ]:
import os
import tempfile
import warnings

import numpy as np
import plotly.graph_objects as go
from sasmodels.core import load_model
from sasmodels.direct_model import DirectModel

import sans_fitter
from sans_fitter import SANSFitter, examples
from sans_fitter.data import RESOLUTION_MODES, ResolutionSetting, apply_resolution

# The fits below are not the point of every cell, so keep their progress
# reporting out of the way. Drop this line to see the full fit output —
# including the resolution each fit reports before it starts.
sans_fitter.set_verbosity('quiet')

## 1. The setting, and what the default means

The mode is **fitter state**, like the model and the parameters, not data
state. It is readable before any data is loaded, and it persists across
`load_data()` / `set_data()`.

In [ ]:
fitter = SANSFitter()

print('modes:  ', RESOLUTION_MODES)
print('default:', fitter.get_resolution())

`'data'` is the default and means *"use whatever this dataset carries"*:

| Mode | Dataset has | Result |
|---|---|---|
| `'data'` | a real `dQ` column | pinhole smearing from that column |
| `'data'` | slit columns only (`dxl`/`dxw`) | slit smearing from those columns |
| `'data'` | no resolution columns | **warns**, evaluates unsmeared |
| `'none'` | anything | unsmeared, file columns ignored |
| `'pinhole'` | anything | `dx = dq_over_q · q` |
| `'slit'` | anything | constant `dxl` (and optional `dxw`) |

Nothing is assumed on your behalf: the one case where the default cannot do
what you asked — a file with no resolution information — warns rather than
quietly picking something.

## 2. What smearing does to a measurement

`sphere` and `sphere_smeared` are two bundled datasets of the *same* sample;
only the second carries a `dQ` column. Smearing washes out the sharp
form-factor minima. This is the signal every mode below is about.

In [ ]:
plain_file = examples.load('sphere')
smeared_file = examples.load('sphere_smeared')

fig = go.Figure()
fig.add_trace(
    go.Scatter(x=plain_file.x, y=plain_file.y, mode='markers',
               marker=dict(size=4), name='no dQ')
)
fig.add_trace(
    go.Scatter(x=smeared_file.x, y=smeared_file.y, mode='markers',
               marker=dict(size=4), name='pinhole dQ')
)
fig.update_layout(
    xaxis_type='log', yaxis_type='log', width=800, height=450,
    xaxis_title='Q (Å⁻¹)', yaxis_title='I(Q)',
    title='Same spheres, measured with and without resolution',
)
fig

## 3. What each mode hands to sasmodels

Under the hood, a mode is applied by writing the resolution **columns** onto a
copy of your dataset. sasmodels picks `Pinhole1D`, `Slit1D` or `Perfect1D` from
exactly those columns, which is why one setting reaches both fitting engines
and DREAM with no engine-specific code.

`apply_resolution()` is public, so we can look at the copy directly and
evaluate the same model through each one.

In [ ]:
TRUTH = {'radius': 60.0, 'scale': 0.02, 'background': 0.001, 'sld': 4.0, 'sld_solvent': 1.0}

# simulate(dq=...) both smears the intensity and attaches the matching dQ
# column, so it behaves like a real measurement at σ_q/q = 10%.
measured = examples.simulate('sphere', dq=0.10, noise=0.02, seed=7, npoints=120, **TRUTH)
kernel = load_model('sphere', dtype='single', platform='dll')

settings = {
    'none — sharp': ResolutionSetting('none'),
    "data — the file's own dQ (10%)": ResolutionSetting('data'),
    'pinhole — a stated 25%': ResolutionSetting('pinhole', dq_over_q=0.25),
    'slit — length 0.05 Å⁻¹': ResolutionSetting('slit', slit_length=0.05),
}

fig = go.Figure()
fig.add_trace(
    go.Scatter(x=measured.x, y=measured.y, mode='markers', name='measured',
               marker=dict(size=4, color='rgb(120,120,120)'), opacity=0.5)
)
for label, setting in settings.items():
    evaluation_copy = apply_resolution(measured, setting)
    curve = np.asarray(DirectModel(evaluation_copy, kernel)(**TRUTH))
    fig.add_trace(go.Scatter(x=measured.x, y=curve, mode='lines', name=label))

fig.update_layout(
    xaxis_type='log', yaxis_type='log', width=850, height=500,
    xaxis_title='Q (Å⁻¹)', yaxis_title='I(Q)',
    title='The same model at the same parameters, under four resolutions',
)
fig

In [ ]:
# Each mode is a different set of columns on a *copy*. Your dataset is untouched.
for label, setting in settings.items():
    copy = apply_resolution(measured, setting)
    dx = 'None' if copy.dx is None else f'max {np.max(copy.dx):.4g}'
    dxl = 'None' if copy.dxl is None else f'{copy.dxl[0]:g}'
    print(f'{label:34s} dx = {dx:12s} dxl = {dxl}')

print(f'\nfitter.data-style original still at dx max {np.max(measured.dx):.4g}')

## 4. What ignoring resolution costs you

The data above was generated at a known radius of 60 Å with a known 10%
pinhole. Fitting it under `'none'` asks a sharp model to reproduce smeared
data, and the optimiser pays for it, mostly in goodness of fit, but in the
radius too.

In [ ]:
def sphere_fitter(data):
    """A sphere fitter on *data*, started well away from the truth."""
    fitter = SANSFitter()
    fitter.set_data(data)
    fitter.set_model('sphere')
    fitter.set_param('radius', value=40.0, min=10.0, max=200.0, vary=True)
    fitter.set_param('scale', value=0.01, min=1e-4, max=1.0, vary=True)
    fitter.set_param('background', value=0.0, min=0.0, max=0.1, vary=True)
    fitter.set_param('sld', value=4.0, vary=False)
    fitter.set_param('sld_solvent', value=1.0, vary=False)
    return fitter


fitters = {}
print(f'truth: radius = {TRUTH["radius"]} Å\n')
for mode in ('data', 'none'):
    fitters[mode] = sphere_fitter(measured)
    fitters[mode].set_resolution(mode)
    result = fitters[mode].fit(engine='bumps', method='amoeba')
    radius = result['parameters']['radius']
    print(
        f"mode '{mode}':  radius = {radius['value']:.3f} ± {radius['stderr']:.3f} Å"
        f"  ({radius['value'] - TRUTH['radius']:+.3f} Å)"
        f"   χ²/dof = {result['chisq']:.3f}"
    )

The radius moves by about 1 Å: seventeen times its own error bar, so it would
be reported as a confident wrong answer. And χ²/dof jumps from ~0.7 to ~4.7:
the residuals below are where you would notice first.

Read that as evidence, not as a dial. Resolution is a property of the
instrument, and smearing is degenerate with real physics: polydispersity
broadens a form-factor minimum much as resolution does. The mode to use is
the one the beamline actually had, not the one that minimises χ².

In [ ]:
fitters['data'].plot_results(show_residuals=True)

In [ ]:
fitters['none'].plot_results(show_residuals=True)

A pinhole width you state yourself is the *same quantity* as the file's `dQ`
column: σ_q/q, so stating the width the data was made with reproduces mode
`'data'`.

In [ ]:
stated = sphere_fitter(measured)
stated.set_resolution('pinhole', dq_over_q=0.10)
from_width = stated.fit(engine='bumps', method='amoeba')
from_column = fitters['data'].fit_result

print(f"pinhole, dq_over_q=0.10: radius = {from_width['parameters']['radius']['value']:.4f} Å")
print(f"data, the file's dQ:     radius = {from_column['parameters']['radius']['value']:.4f} Å")

## 5. Stating a width yourself

### A file with no `dQ` column

Mode `'data'` has nothing to use here, so it warns and evaluates unsmeared.


In [ ]:
plain = examples.simulate('sphere', noise=0.02, seed=7, npoints=120, **TRUTH)
print(f'dx column: {plain.dx}')

fitter = sphere_fitter(plain)
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always')
    fitter.fit(engine='bumps', method='amoeba')
for warning in caught:
    print(f'\nwarning: {warning.message}')

Two ways to silence it:

- `set_resolution('none')` — *"this measurement is sharp"*
- `set_resolution('pinhole', dq_over_q=...)` — *"the instrument smears by this"*

Here the first is the truth and the second is a guess. Smearing data that was
never smeared biases the radius just as surely as ignoring real smearing did,
in the opposite direction. **The mode has to match the measurement**, which for
a real instrument is a question for the beamline rather than a number to guess.

In [ ]:
for label, mode, kwargs in (
    ("mode 'none' (correct here)", 'none', {}),
    ('a guessed 10% pinhole', 'pinhole', {'dq_over_q': 0.10}),
):
    fitter = sphere_fitter(plain)
    fitter.set_resolution(mode, **kwargs)
    result = fitter.fit(engine='bumps', method='amoeba')
    radius = result['parameters']['radius']
    print(
        f'{label:28s} radius = {radius["value"]:.3f} Å '
        f'({radius["value"] - TRUTH["radius"]:+.3f} Å)   '
        f'χ²/dof = {result["chisq"]:.3f}'
    )

### Slit geometry (USANS)

`slit_length` is the slit dimension along q: an **absolute** width in Å⁻¹, not
a relative one, and it is required: sasmodels smears along the slit length, so
a width alone has nothing to integrate over. Omit `slit_width` for the usual
long-slit geometry.

The pinhole data from §4 is the wrong data for a slit.

In [ ]:
fitter = sphere_fitter(measured)
fitter.set_resolution('slit', slit_length=0.05)
print(fitter.get_resolution())

result = fitter.fit(engine='bumps', method='amoeba')
print(f"\nradius = {result['parameters']['radius']['value']:.3f} Å")
print(f"χ²/dof = {result['chisq']:.3f}  — which is what a bad fit is for")

## 6. Provenance

### Your dataset is never modified

The setting is applied to a copy made for evaluation, so plots, CSV export,
P(r) inversion and `data_ops` all keep seeing the dataset you loaded.

In [ ]:
fitter = sphere_fitter(measured)
fitter.set_resolution('pinhole', dq_over_q=0.25)
fitter.fit(engine='bumps', method='amoeba')

print('fit ran at:        σ_q/q = 0.25')
print(f'fitter.data still: σ_q/q = {np.max(fitter.data.dx) / np.max(fitter.data.x):.2f}')
print(f'untouched:         {np.allclose(fitter.data.dx, measured.dx)}')

### The choice is recorded with the fit

A fitted parameter set only means something alongside the smearing that
produced it, so `save_results()` writes the mode into the CSV header.

In [ ]:
with tempfile.TemporaryDirectory() as folder:
    path = os.path.join(folder, 'fit.csv')
    fitter.save_results(path)
    with open(path, encoding='utf-8') as handle:
        print(''.join(line for line in handle if line.startswith('# ') and 'Resolution' in line))

### Guardrails

Every check runs before any state is touched, so a rejected call leaves the
fitter exactly as it was.

In [ ]:
fitter = sphere_fitter(measured)
fitter.set_resolution('pinhole', dq_over_q=0.10)

bad_calls = [
    ('gaussian', {}),                  # not one of the four modes
    ('pinhole', {}),                   # a pinhole needs a width
    ('slit', {'slit_width': 0.01}),    # a slit needs a length
    ('data', {'dq_over_q': 0.1}),      # 'data' takes no width at all
    ('pinhole', {'dq_over_q': 0}),     # a zero width is 'none', not a pinhole
]
for mode, kwargs in bad_calls:
    arguments = ''.join(f', {name}={value!r}' for name, value in kwargs.items())
    try:
        fitter.set_resolution(mode, **kwargs)
    except ValueError as error:
        print(f"set_resolution('{mode}'{arguments})\n  -> {error}\n")

print(f'still in force: {fitter.get_resolution()}')

## Summary

- Resolution is a **stated choice**: `set_resolution(...)` / `get_resolution()`,
  four modes mirroring SasView's Fit Page.
- The default `'data'` uses the file's own columns and **warns** rather than
  guessing when there are none.
- The mode must match the measurement. Ignoring real smearing and inventing
  smearing that is not there are the same mistake in opposite directions, and
  χ²/dof catches both, since smearing is degenerate with real physics such
  as polydispersity.
- `dq_over_q` is **σ_q/q, a Gaussian 1-σ, not FWHM** (divide a quoted FWHM by
  about 2.355). `slit_length` / `slit_width` are **absolute** widths in Å⁻¹.
- Your dataset is never modified, and the setting reaches every engine:
  bumps, lmfit and DREAM.


See also `examples/resolution_example.py` for the same content as a script, and
the *Resolution (Smearing)* section of `docs/usage.md`.